**Trabajo de Titulación**

Desarrollo de un módulo de detección de zonas de acceso terrestre en entornos de desastres naturales mediante segmentación de imágenes aéreas.

**Autora:** Karen Doménica Lara Félix

In [ ]:
# Montaje del almacenamiento en Google Drive

from google.colab import drive
drive.mount('/content/drive')

# Librerías para manipulación de archivos y procesamiento de imágenes

import os
import cv2
import numpy as np
from tqdm import tqdm

# Dimensiones cuadradas objetivo para la entrada de las redes (256x256 px)

IMG_SIZE = 256

# Definición de rutas principales del conjunto de datos con una división de 70,15,15 para train, validation, test respectivamente

BASE_DIR = "/content/drive/MyDrive/Proyecto_Vias/Dataset_Dividido"

TRAIN_IMG_DIR = os.path.join(BASE_DIR, "Train/img")
TRAIN_MASK_DIR = os.path.join(BASE_DIR, "Train/mask")

VAL_IMG_DIR = os.path.join(BASE_DIR, "Validation/img")
VAL_MASK_DIR = os.path.join(BASE_DIR, "Validation/mask")

TEST_IMG_DIR = os.path.join(BASE_DIR, "Test/img")
TEST_MASK_DIR = os.path.join(BASE_DIR, "Test/mask")


# Directorio de salida para guardar las matrices procesadas

SAVE_DIR = "/content/drive/MyDrive/Proyecto_Vias/Dataset_NPY"

os.makedirs(SAVE_DIR, exist_ok=True)

# FUNCIÓN RESIZE + PADDING

# Redimensiona la imagen manteniendo su relación de aspecto original
# y añade bordes negros (padding) para completar el tamaño objetivo.


def resize_with_padding(image, target_size):

    h, w = image.shape[:2]

    # Cálculo del factor de escala según el lado mayor
    scale = target_size / max(h, w)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(image, (new_w, new_h))

    # Creación del lienzo negro según el número de canales
    if len(image.shape) == 3:

        padded = np.zeros((target_size, target_size, 3), dtype=np.uint8)

    else:

        padded = np.zeros((target_size, target_size), dtype=np.uint8)

    # Centrado de la imagen redimensionada sobre el lienzo
    x_offset = (target_size - new_w) // 2
    y_offset = (target_size - new_h) // 2

    padded[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

    return padded


# FUNCIÓN PARA CARGAR DATASET

#Carga, preprocesa y normaliza las imágenes y sus respectivas máscaras.

def load_dataset(img_dir, mask_dir):

    images = []
    masks = []

    # Filtrado de archivos compatibles
    image_files = sorted(os.listdir(img_dir))

    image_files = [
        f for f in image_files
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ]

    print(f"\nCargando {img_dir}")
    print(f"Total imágenes: {len(image_files)}")

    for file_name in tqdm(image_files):

        # Asociación de rutas para imagen y su correspondiente máscara en PNG

        img_path = os.path.join(img_dir, file_name)

        mask_name = os.path.splitext(file_name)[0] + ".png"
        mask_path = os.path.join(mask_dir, mask_name)

        # LEER IMAGEN
        # Carga y conversión de espacio de color BGR a RGB

        image = cv2.imread(img_path)

        if image is None:
            print(f"Error imagen: {img_path}")
            continue

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = resize_with_padding(image, IMG_SIZE)

        # Normalizar píxeles al rango [0, 1]
        image = image.astype(np.float32) / 255.0

        # Leer máscara en escala de grises
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            print(f"Error máscara: {mask_path}")
            continue

        mask = resize_with_padding(mask, IMG_SIZE)

        # Binarización explícita (umbral 127) y ajuste de dimensiones para la red
        mask = (mask > 127).astype(np.float32)

        # Agregar canal
        mask = np.expand_dims(mask, axis=-1)

        images.append(image)
        masks.append(mask)

    return np.array(images), np.array(masks)

# CARGAR TRAIN

X_train, y_train = load_dataset(TRAIN_IMG_DIR,TRAIN_MASK_DIR)
print("Train:", X_train.shape, y_train.shape)

# CARGAR VALIDATION

X_val, y_val = load_dataset(VAL_IMG_DIR,VAL_MASK_DIR)

print("Validation:", X_val.shape, y_val.shape)

# CARGAR TEST

X_test, y_test = load_dataset(TEST_IMG_DIR,TEST_MASK_DIR)

print("Test:", X_test.shape, y_test.shape)

# Guardado de arreglos NumPy para agilizar lecturas posteriores durante el entrenamiento

print("\nGuardando archivos .npy...")

np.save(os.path.join(SAVE_DIR, "X_train.npy"), X_train)
np.save(os.path.join(SAVE_DIR, "y_train.npy"), y_train)

np.save(os.path.join(SAVE_DIR, "X_val.npy"), X_val)
np.save(os.path.join(SAVE_DIR, "y_val.npy"), y_val)

np.save(os.path.join(SAVE_DIR, "X_test.npy"), X_test)
np.save(os.path.join(SAVE_DIR, "y_test.npy"), y_test)

print("\nPreprocesamiento completado correctamente.")
print(f"Archivos guardados en:\n{SAVE_DIR}")



Mounted at /content/drive

Cargando /content/drive/MyDrive/Proyecto_Vias/Dataset_Dividido/Train/img
Total imágenes: 703


100%|██████████| 703/703 [07:37<00:00,  1.54it/s]


Train: (703, 256, 256, 3) (703, 256, 256, 1)

Cargando /content/drive/MyDrive/Proyecto_Vias/Dataset_Dividido/Validation/img
Total imágenes: 151


100%|██████████| 151/151 [01:23<00:00,  1.82it/s]


Validation: (151, 256, 256, 3) (151, 256, 256, 1)

Cargando /content/drive/MyDrive/Proyecto_Vias/Dataset_Dividido/Test/img
Total imágenes: 151


100%|██████████| 151/151 [01:24<00:00,  1.78it/s]


Test: (151, 256, 256, 3) (151, 256, 256, 1)

Guardando archivos .npy...

Preprocesamiento completado correctamente.
Archivos guardados en:
/content/drive/MyDrive/Proyecto_Vias/Dataset_NPY
